In [27]:
import json
from scipy.optimize import linear_sum_assignment
from datasets import load_dataset
from datasets import Dataset
from PIL import Image
from scipy.optimize import linear_sum_assignment
import numpy as np
import re

def smart_resize(image: Image.Image, max_size=640): 
    """Resize proportionally so the longest side equals max_size.""" 
    w, h = image.size 
    scale = max_size / max(w, h) 
    new_size = (int(w * scale), int(h * scale)) 
    return image.resize(new_size, Image.LANCZOS)
    
def compute_iou(boxA, boxB):
    """Compute IoU via Hungarian algorithm to match bounding box with different orders
    """
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0:
        return 0.0
    boxA_area = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxB_area = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = boxA_area + boxB_area - inter
    return inter / union if union > 0 else 0

def normalize_detections(dets):
    """
    Convert model outputs into a standard format:
    [{"bbox": [...], "label": "..."}]
    """
    if not isinstance(dets, list):
        return None

    normalized = []
    for d in dets:
        if not isinstance(d, dict):
            continue

        bbox = d.get("bbox") or d.get("bbox_2d")
        label = d.get("label")

        if bbox is None or label is None:
            continue

        if len(bbox) != 4:
            continue

        normalized.append({
            "bbox": [float(x) for x in bbox],
            "label": str(label)
        })

    return normalized if normalized else None

def parse_detection_list(output):
    """
    Parse model output containing a list of detections.
    Accepts bbox or bbox_2d.
    """
    try:
        data = json.loads(output)
        return normalize_detections(data)
    except Exception:
        pass

    pattern = r'\{\s*"bbox(?:_2d)?"\s*:\s*\[([0-9\.,\s]+)\]\s*,\s*"label"\s*:\s*"([^"]+)"\s*\}'
    matches = re.findall(pattern, output)

    results = []
    for bbox_str, label in matches:
        nums = [float(x) for x in bbox_str.split(",")]
        if len(nums) == 4:
            results.append({"bbox": nums, "label": label})

    return results if results else []


def parse_ground_truth(gt_string):
    parsed_gt = parse_detection_list(
        gt_string,
    )
    if parsed_gt == None :
        parsed_gt = []
    return parsed_gt

def parse_prediction(completion):
    text = completion[0]["content"]
    return parse_detection_list(
        text
    )


def reward_parseable(completion):
    pred = parse_prediction(completion)
    if pred is None:
        return 0.0
    return 1.0

def reward_object_count(completion,info):
    pred = parse_prediction(completion)
    if pred is None:
        return 0.0

    gt = parse_ground_truth(info["gt"])
    return 1.0 / (1 + abs(len(pred) - len(gt)))

def reward_matching(completion,info):
    pred = parse_prediction(completion)
    if pred is None:
        return 0.0

    gt = parse_ground_truth(info["gt"])
    if len(pred) == 0 or len(gt) == 0:
        return 0.0

    n, m = len(pred), len(gt)
    cost = np.zeros((n, m))

    for i, p in enumerate(pred):
        for j, g in enumerate(gt):
            iou = compute_iou(p["bbox"], g["bbox"])
            label_correct = 1.0 if p["label"] == g["label"] else 0.0
            cost[i, j] = -(iou + 0.5 * label_correct)

    row_ind, col_ind = linear_sum_assignment(cost)

    scores = []
    for r, c in zip(row_ind, col_ind):
        p = pred[r]
        g = gt[c]
        iou = compute_iou(p["bbox"], g["bbox"])
        label_ok = 1.0 if p["label"] == g["label"] else 0.0
        scores.append(0.7 * iou + 0.3 * label_ok)

    return float(np.mean(scores))

def reward_parseables(completions,info,**kwargs) :
    return [reward_parseable(completion) for completion in completions]

def reward_object_counts(completions,info,**kwargs) :
    return [reward_object_count(completion,inf) for completion, inf in zip(completions, info)]

def reward_matchings(completions,info,**kwargs) :
    return [reward_matching(completion,inf) for completion, inf in zip(completions, info)]

In [8]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from datasets import load_dataset

model_name = "Qwen/Qwen3-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(model_name)

SYSTEM_PROMPT = (
    ""
)

category = "window"


def preprocess_fn(example):
    img = example["image"]
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    img = img.convert("RGB")
    img = smart_resize(img, 640)

    gt = example["ground_truth"]  # bounding box list

    prompt = [{
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f'locate every instance that belongs to the following categories: {category}. '
                        f'For each window, report bbox coordinates, in JSON format like this: '
                        f'{{"bbox_2d": [x1, y1, x2, y2], "label": "{category}"}}'
            },
            {"type": "image"}
        ]
    }]
    #prompt = processor.apply_chat_template(prompt, add_generation_prompt=True)
    return {
        "prompt": prompt,
        "answer": json.dumps(gt),  # not used directly; reward compares raw
        "images": [img],
        "info": {"gt": gt}
    }

dataset_id = 'UlrickBL/elevation-dataset'
train_dataset = load_dataset(dataset_id, split='train')

train_dataset = train_dataset.map(preprocess_fn)

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

In [9]:
train_dataset

Dataset({
    features: ['id', 'image', 'ground_truth', 'prompt', 'answer', 'images', 'info'],
    num_rows: 22
})

In [4]:
import torch
from peft import LoraConfig, get_peft_model

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)


lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["qkv","gate_proj","up_proj","base_layer"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

trainable params: 8,912,896 || all params: 2,136,444,928 || trainable%: 0.4172


In [5]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 9.2 MB/s eta 0:00:00a 0:00:01


In [ ]:
from trl import GRPOConfig,GRPOTrainer

training_args = GRPOConfig(
    output_dir="Qwen3VL-2B-BBOX-Window",
    learning_rate=3e-5,
    temperature = 0.7,
    lr_scheduler_type = "cosine",
    warmup_steps = 30,
    beta = 0.01,
    save_strategy = "steps",
    remove_unused_columns=False,
    bf16=True,

    per_device_train_batch_size=2,
    max_completion_length=1024,
    num_generations=2,
    max_prompt_length=2048,

    report_to=["tensorboard"],
    logging_steps=10,
    save_steps=10,
    reward_weights = [0.2, 0.2, 0.6],
)

trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=[reward_parseables,
            reward_object_counts,
            reward_matchings],
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

<string>:192: FutureWarning: The `max_prompt_length` argument is deprecated and will be removed in version 0.28.0. You should instead filter your dataset before training to ensure that prompts do not exceed your desired length.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
